# 06 — Interpretability & Case Studies

Port-level risk timelines and conformal uncertainty bands for the top-disrupted storms.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.models.portex_glm import PORTEXWeightedGLM
from src.evaluation.conformal import RollingConformalWrapper
from src.data.preprocessing import FEATURE_COLS, GROUP_COL, LABEL_COL

df = pd.read_parquet("../data/processed/portex_labeled.parquet")
df["risk_norm"] = df["PORTEX_risk_recalc"] / 100.0
df = df.sort_values(["SID", "PORT", "DATE"])

model = PORTEXWeightedGLM(penalty="l1")
model.fit(df[FEATURE_COLS], df[LABEL_COL])


In [ ]:
# ── Top-3 storms by event count ──────────────────────────
top_storms = (df.groupby("SID")[LABEL_COL].sum()
               .sort_values(ascending=False).index[:3])
print("Top disruption storms:", list(top_storms))


In [ ]:
# ── Risk timeline with conformal band ────────────────────
def plot_storm_risk(storm_id, port, alpha=0.10, window=50):
    sub = df[(df["SID"] == storm_id) & (df["PORT"] == port)].copy()
    if len(sub) < 5:
        return
    probas = model.predict_proba(sub[FEATURE_COLS])[:, 1]
    labels = sub[LABEL_COL].values

    cp = RollingConformalWrapper(model, alpha=alpha, window=window)
    lower, upper = [], []
    for p, y in zip(probas, labels):
        _, q_hat = cp.predict_set(p)
        lower.append(max(0, p - q_hat))
        upper.append(min(1, p + q_hat))
        cp.update_calibration(p, int(y))

    fig, ax = plt.subplots(figsize=(10, 3.5))
    idx = range(len(probas))
    ax.fill_between(idx, lower, upper, alpha=0.25, color="#9C27B0", label=f"{int((1-alpha)*100)}% conf band")
    ax.plot(idx, probas, color="#2196F3", linewidth=1.5, label="Risk score")
    ax.scatter([i for i, y in enumerate(labels) if y == 1],
               [probas[i] for i, y in enumerate(labels) if y == 1],
               color="crimson", zorder=5, s=40, label="Disruption event")
    ax.set(xlabel="Advisory index", ylabel="Risk probability",
           title=f"Storm {storm_id} — Port {port}")
    ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"../results/figures/06_case_{storm_id}_{port}.png", dpi=150)
    plt.show()

for storm in top_storms:
    for port in df["PORT"].unique():
        plot_storm_risk(storm, port)
